# Loan Analysis - Data Preprocessing

**Section Goals:**
> - Remove or fill any missing data.
> - Remove unnecessary or repetitive features.
> - Convert categorical string features to dummy variables.
> - Split data into train/test sets.
> - Remove outliers from training data.
> - Normalize/scale the features.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 50)

In [ ]:
# Load the full accepted dataset and filter to 2012-2014, Fully Paid / Charged Off only
data = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv.gz", low_memory=False)

# Keep only the columns we need (matching lending_club_loan_two) + desc for LLM
keep_cols = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
    'verification_status', 'issue_d', 'loan_status', 'purpose', 'title',
    'dti', 'earliest_cr_line', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'initial_list_status', 'application_type',
    'mort_acc', 'pub_rec_bankruptcies', 'zip_code', 'addr_state', 'desc'
]
data = data[keep_cols].copy()

# Filter to final loan statuses only
data = data[data['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()

# Filter to 2012-2014
data['issue_d'] = pd.to_datetime(data['issue_d'], format='%b-%Y')
data = data[(data['issue_d'].dt.year >= 2012) & (data['issue_d'].dt.year <= 2014)].copy()

# Encode target
data['loan_status'] = data.loan_status.map({'Fully Paid': 1, 'Charged Off': 0})

# Convert dates
data['earliest_cr_line'] = pd.to_datetime(data['earliest_cr_line'], format='%b-%Y')

# Handle int_rate if stored as string with '%'
if data['int_rate'].dtype == object:
    data['int_rate'] = data['int_rate'].str.replace('%', '').astype(float)

# Consolidate rare home_ownership categories
data.loc[(data.home_ownership == 'ANY') | (data.home_ownership == 'NONE'), 'home_ownership'] = 'OTHER'

# Binarize pub_rec, mort_acc, pub_rec_bankruptcies
data['pub_rec'] = data['pub_rec'].apply(lambda x: 1 if x > 0 else 0)
data['mort_acc'] = data['mort_acc'].apply(lambda x: 1 if x >= 1.0 else (0 if x == 0.0 else x))
data['pub_rec_bankruptcies'] = data['pub_rec_bankruptcies'].apply(lambda x: 1 if x >= 1.0 else (0 if x == 0.0 else x))

# Store desc separately before dropping it for ML preprocessing
desc_series = data[['desc']].copy()
desc_series.index = data.index
data.drop('desc', axis=1, inplace=True)

print(f"Data shape (2012-2014, Fully Paid/Charged Off): {data.shape}")
print(f"Rows with desc: {desc_series['desc'].notna().sum()} ({desc_series['desc'].notna().mean()*100:.1f}%)")
data.head()

## Missing Values

In [3]:
for column in data.columns:
    if data[column].isna().sum() != 0:
        missing = data[column].isna().sum()
        portion = (missing / data.shape[0]) * 100
        print(f"'{column}': number of missing values '{missing}' ==> {portion:.3f}%")

'emp_title': number of missing values '22927' ==> 5.789%
'emp_length': number of missing values '18301' ==> 4.621%
'title': number of missing values '1756' ==> 0.443%
'revol_util': number of missing values '276' ==> 0.070%
'mort_acc': number of missing values '37795' ==> 9.543%
'pub_rec_bankruptcies': number of missing values '535' ==> 0.135%


### `emp_title`

Realistically there are too many unique job titles to try to convert this to a dummy variable feature. Let's remove the `emp_title` column.

In [4]:
print(f"Unique emp_title values: {data.emp_title.nunique()}")
data.drop('emp_title', axis=1, inplace=True)

Unique emp_title values: 173105


### `emp_length`

Charge off rates are extremely similar across all employment lengths. So we are going to drop the `emp_length` column.

In [5]:
data.emp_length.unique()

array(['10+ years', '4 years', '< 1 year', '6 years', '9 years',
       '2 years', '3 years', '8 years', '7 years', '5 years', '1 year',
       nan], dtype=object)

In [6]:
for year in data.emp_length.dropna().unique():
    print(f"{year}:")
    vals = data[data.emp_length == year].loan_status.value_counts()
    print(f"  Charged Off rate: {(vals.get(0, 0) / vals.sum()) * 100:.2f}%")
    print()

10+ years:
  Charged Off rate: 18.42%

4 years:
  Charged Off rate: 19.24%

< 1 year:
  Charged Off rate: 20.69%

6 years:
  Charged Off rate: 18.92%

9 years:
  Charged Off rate: 20.05%

2 years:
  Charged Off rate: 19.33%

3 years:
  Charged Off rate: 19.52%

8 years:
  Charged Off rate: 19.98%

7 years:
  Charged Off rate: 19.48%

5 years:
  Charged Off rate: 19.22%

1 year:
  Charged Off rate: 19.91%



In [7]:
data.drop('emp_length', axis=1, inplace=True)

### `title`

The title column is simply a string subcategory/description of the purpose column. So we are going to drop the title column.

In [8]:
data.drop('title', axis=1, inplace=True)

### `mort_acc`

Fill missing `mort_acc` values using the mean per `total_acc` group.

In [9]:
print(f"Missing mort_acc: {data.mort_acc.isna().sum()}")
data.corr(numeric_only=True)['mort_acc'].drop('mort_acc').sort_values()

Missing mort_acc: 37795


int_rate               -0.09
dti                    -0.02
revol_util              0.00
pub_rec                 0.05
pub_rec_bankruptcies    0.05
loan_status             0.07
open_acc                0.12
revol_bal               0.17
installment             0.18
annual_inc              0.20
loan_amnt               0.21
total_acc               0.30
Name: mort_acc, dtype: float64

In [10]:
total_acc_avg = data.groupby('total_acc')['mort_acc'].mean()

def fill_mort_acc(total_acc, mort_acc):
    if np.isnan(mort_acc):
        return total_acc_avg.get(total_acc, total_acc_avg.mean()).round()
    else:
        return mort_acc

data['mort_acc'] = data.apply(lambda x: fill_mort_acc(x['total_acc'], x['mort_acc']), axis=1)

### Remaining missing values

These features have missing data points, but they account for less than 0.5% of the total data. So we are going to drop the rows with missing values.

In [11]:
for column in data.columns:
    if data[column].isna().sum() != 0:
        missing = data[column].isna().sum()
        portion = (missing / data.shape[0]) * 100
        print(f"'{column}': number of missing values '{missing}' ==> {portion:.3f}%")

'revol_util': number of missing values '276' ==> 0.070%
'pub_rec_bankruptcies': number of missing values '535' ==> 0.135%


In [12]:
data.dropna(inplace=True)
print(f"Shape after dropping NAs: {data.shape}")

Shape after dropping NAs: (395219, 24)


## Categorical Variables and Dummy Variables

In [13]:
print([column for column in data.columns if data[column].dtype == object])

['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type', 'address']


### `term`

In [14]:
data.term.unique()

array([' 36 months', ' 60 months'], dtype=object)

In [15]:
term_values = {' 36 months': 36, ' 60 months': 60}
data['term'] = data.term.map(term_values)
data.term.unique()

array([36, 60])

### `grade` & `sub_grade`

We know that `grade` is just a parent feature of `sub_grade`, so we are going to drop it.

In [16]:
data.drop('grade', axis=1, inplace=True)

In [17]:
dummies = ['sub_grade', 'verification_status', 'purpose', 'initial_list_status',
           'application_type', 'home_ownership']
data = pd.get_dummies(data, columns=dummies, drop_first=True)

### `zip_code` & `addr_state`

Drop both — `zip_code` creates too many sparse dummies and `addr_state` adds 49 features with limited predictive value.

In [ ]:
data.drop(['zip_code', 'addr_state'], axis=1, inplace=True)
print(f"Shape after dropping zip_code & addr_state: {data.shape}")

### `issue_d`

This would be data leakage — we wouldn't know beforehand whether or not a loan would be issued when using our model, so in theory we wouldn't have this feature. Let's drop it.

In [19]:
data.drop('issue_d', axis=1, inplace=True)

### `earliest_cr_line`

This appears to be a historical timestamp feature. Extract the year from this feature.

In [20]:
data['earliest_cr_line'] = data.earliest_cr_line.dt.year
print(f"Unique years: {data.earliest_cr_line.nunique()}")
data.earliest_cr_line.value_counts().head(10)

Unique years: 65


earliest_cr_line
2000    29302
2001    29031
1999    26444
2002    25849
2003    23623
1998    22695
2004    20875
1997    18702
1996    18362
2005    17388
Name: count, dtype: int64

## Train Test Split

In [21]:
w_p = data.loan_status.value_counts()[0] / data.shape[0]
w_n = data.loan_status.value_counts()[1] / data.shape[0]

print(f"Weight of positive values (Charged Off): {w_p:.4f}")
print(f"Weight of negative values (Fully Paid): {w_n:.4f}")

Weight of positive values (Charged Off): 0.1962
Weight of negative values (Fully Paid): 0.8038


In [22]:
train, test = train_test_split(data, test_size=0.33, random_state=42)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (264796, 79)
Test shape: (130423, 79)


## Removing Outliers

We remove outliers only from the training set to avoid data leakage.

In [23]:
print(f"Train shape before outlier removal: {train.shape}")
train = train[train['annual_inc'] <= 250000]
train = train[train['dti'] <= 50]
train = train[train['open_acc'] <= 40]
train = train[train['total_acc'] <= 80]
train = train[train['revol_util'] <= 120]
train = train[train['revol_bal'] <= 250000]
print(f"Train shape after outlier removal: {train.shape}")

Train shape before outlier removal: (264796, 79)
Train shape after outlier removal: (262143, 79)


## Normalizing the Data

In [24]:
X_train, y_train = train.drop('loan_status', axis=1), train.loan_status
X_test, y_test = test.drop('loan_status', axis=1), test.loan_status

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (262143, 78), y_train: (262143,)
X_test: (130423, 78), y_test: (130423,)


In [25]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
# Convert to float32 for model compatibility
X_train_scaled = np.array(X_train_scaled).astype(np.float32)
X_test_scaled = np.array(X_test_scaled).astype(np.float32)
y_train = np.array(y_train).astype(np.float32)
y_test = np.array(y_test).astype(np.float32)

print(f"X_train dtype: {X_train_scaled.dtype}")
print(f"y_train dtype: {y_train.dtype}")

X_train dtype: float32
y_train dtype: float32


In [ ]:
# Save processed data for ML
np.savez("../data/processed/02_processed_data.npz",
         X_train=X_train_scaled, X_test=X_test_scaled,
         y_train=y_train, y_test=y_test)
joblib.dump(scaler, "../data/processed/02_scaler.joblib")

print("Saved processed data and scaler.")
print(f"Number of features: {X_train_scaled.shape[1]}")

## LLM Evaluation Sample

Export 100 observations from the test set (stratified by class) with original feature names and borrower description for LLM evaluation.

In [ ]:
# Build LLM sample from the test set rows (before scaling) with desc
# Use the original test DataFrame (pre-scaling) to keep readable feature names
llm_pool = test.copy()
llm_pool['desc'] = desc_series.reindex(llm_pool.index)['desc']

# Only keep rows that have a description
llm_with_desc = llm_pool[llm_pool['desc'].notna()].copy()

# Stratified sample of 100 rows matching test set class distribution
n_sample = 100
class_dist = llm_with_desc['loan_status'].value_counts(normalize=True)
print(f"Class distribution in pool: {dict(class_dist.round(3))}")

samples = []
for label, frac in class_dist.items():
    n = round(n_sample * frac)
    pool = llm_with_desc[llm_with_desc['loan_status'] == label]
    samples.append(pool.sample(n=n, random_state=42))

llm_sample = pd.concat(samples).sample(frac=1, random_state=42)  # shuffle

llm_sample.to_csv("../data/processed/02_llm_sample.csv", index=False)
print(f"LLM sample saved: {llm_sample.shape}")
print(f"Class distribution in sample:\n{llm_sample['loan_status'].value_counts()}")